In [1]:
import pandas as pd
from pathlib import Path
import sys
from IPython.display import display, Markdown

In [2]:
sys.path.append("../src")
from explainer import ForensicExplainer
from models import NetworkRoleClassifier

In [3]:
BASE_DIR = Path.cwd().parent
CHATS_CSV = BASE_DIR / "data" / "processed" / "chats_completos.csv"
NODES_FEATURES_CSV = BASE_DIR / "data" / "processed" / "nodos_features.csv"
ANOMALIES_CSV = BASE_DIR / "data" / "processed" / "chats_top.csv"

In [4]:
explainer = ForensicExplainer(model_name="llama-3.3-70b-versatile")

df_chats = pd.read_csv(CHATS_CSV)
df_features = pd.read_csv(NODES_FEATURES_CSV)
df_anomalies = pd.read_csv(ANOMALIES_CSV)

In [5]:
clasificador = NetworkRoleClassifier(n_clusters=4)
features = ['In_Degree', 'Out_Degree', 'Betweenness', 'PageRank', 'Hub_Score', 'Authority_Score']
df_model, _, _ = clasificador.fit_predict(df_features, features)

In [ ]:
# SECCIÓN 1: EXPLICABILIDAD DE GRAFOS
display(Markdown("## 🧠 1. Análisis Estructural y Perfilado de Actores Clave"))
display(Markdown("Evaluación semántica de los 5 nodos con mayor poder jerárquico (PageRank) en la red."))

# Filtramos los 5 actores principales
top_actores = df_model.nlargest(5, 'PageRank')

for _, row in top_actores.iterrows():
    actor = row['Nodo']
    rol = row.get('Etiqueta_Forense', 'Desconocido')
    pr_score = row['PageRank']
    
    # Filtramos mensajes del actor
    df_actor = df_chats[df_chats['Emisor'].str.upper().str.contains(str(actor).upper(), na=False)]
    
    mensajes_muestra = []
    # Ordenamos por longitud para coger los más sustanciales
    for _, msg_row in df_actor.sort_values(by='Mensaje', key=lambda x: x.str.len(), ascending=False).head(5).iterrows():
        # Formateamos incluyendo la emoción
        mensajes_muestra.append(f"[Emoción: {msg_row['NLP_Emocion']}] - {msg_row['Mensaje']}")
        
    if mensajes_muestra:
        explicacion = explainer.explain_actor_profile(actor, rol, pr_score, "\n".join(mensajes_muestra))
        display(Markdown(f"### 👤 Actor: `{actor}`"))
        display(Markdown(f"- **Jerarquía Matemática:** `{rol}` (PageRank: {pr_score:.4f})"))
        display(Markdown(f"- **Evaluación Forense (LLM + NLP):**\n> {explicacion}\n"))
        display(Markdown("---"))

## 🧠 1. Análisis Estructural y Perfilado de Actores Clave

Evaluación semántica de los 5 nodos con mayor poder jerárquico (PageRank) en la red.

### 👤 Actor: `RODOLFO REYES`

- **Clasificación Matemática:** `Cúpula Directiva (Líderes)`

- **Evaluación Forense (LLM):**
> El contenido de los mensajes de Rodolfo REYES concuerda con su rol de Cúpula Directiva, ya que maneja información confidencial y estratégica sobre ayudas públicas y préstamos. Su nivel de jerarquía es alto, ya que interactúa con personas influyentes y toma decisiones clave. La información que maneja es sensible y revela una red de influencias compleja. Su posición de liderazgo es evidente en las conversaciones.


...

---

### 👤 Actor: `JULIO MARTINEZ MARTINEZ`

- **Clasificación Matemática:** `Operadores Periféricos / Ejecutores`

- **Evaluación Forense (LLM):**
> El contenido de los mensajes de JULIO MARTINEZ MARTINEZ sugiere una comunicación concisa y directa, propia de un operador periférico. Maneja información relacionada con empresas y negocios, y su interacción indica un nivel de jerarquía moderado, con acceso a información estratégica y contacto con altos directivos. La mención a un presidente y la aprobación de decisiones refuerzan esta conclusión. Su rol parece concordar con el clasificado.


---

In [ ]:
# SECCIÓN 2: EXPLICABILIDAD DE ANOMALÍAS TEMPORALES
display(Markdown("---"))
display(Markdown("## 🚨 2. Análisis Semántico de Anomalías de Comportamiento"))
display(Markdown("Evaluación de los 5 eventos temporales más atípicos detectados por el algoritmo Isolation Forest."))

# Ordenamos por Anomaly_Score (los más negativos son los más anómalos)
df_anom_enrich = pd.merge(df_anomalies, df_chats[['Datetime', 'Mensaje', 'NLP_Emocion']], on=['Datetime', 'Mensaje'], how='left')
top_anomalias = df_anom_enrich.sort_values(by='Anomaly_Score', ascending=True).head(5)

for i, row in top_anomalias.iterrows():
    fecha = str(row['Datetime'])
    hora = f"{row['Hour']}:00"
    texto = str(row['Mensaje'])
    longitud = row['Message_Length']
    score = row['Anomaly_Score']
    emocion = str(row.get('NLP_Emocion', 'Neutro'))
    
    explicacion = explainer.explain_anomaly(fecha, hora, longitud, score, emocion, texto)
    
    display(Markdown(f"### ⚠️ Anomalía Crítica Nivel {i+1}"))
    display(Markdown(f"- **Metadatos:** `Hora: {hora} | Score: {score:.3f} | Emoción NLP: {emocion.upper()}`"))
    display(Markdown(f"- **Texto Base:** _{texto[:150]}..._"))
    display(Markdown(f"- **Evaluación Forense (LLM + NLP):**\n> {explicacion}\n"))
    display(Markdown("---"))

---

## 🚨 2. Análisis Semántico de Anomalías de Comportamiento

Evaluación de los 5 eventos temporales más atípicos detectados por el algoritmo Isolation Forest.

### ⚠️ Anomalía Crítica Nivel 1

- **Metadatos:** `Fecha: 2020-11-04 09:15:19 | Score: -0.056 | Longitud: 1149 chars`

...

### ⚠️ Anomalía Crítica Nivel 5

- **Metadatos:** `Fecha: 2016-07-21 01:53:51 | Score: -0.034 | Longitud: 23 chars`

- **Texto Base:** _Moncho Gordils por aqui..._

- **Evaluación Forense (LLM):**
> El mensaje "Moncho Gordils por aqui" presenta una longitud inusual de 23 caracteres, lo que sugiere una posible codificación o mensaje cifrado. La marcación como anomalía matemática extrema por el modelo de Isolation Forest indica un patrón atípico. El contexto y el contenido del mensaje podrían sugerir una coordinación clandestina o una señal de encuentro. La evaluación forense sugiere que este evento requiere una investigación más profunda para determinar su naturaleza y posible impacto.


---